In [2]:
import os

def get_data(game):
    directory = f'games_descriptions/{game}/scores_files'
    config_file = f'games_descriptions/{game}/config.txt'

    # Read the config file
    with open(config_file, 'r') as file:
        config_lines = file.readlines()

    # Parse the config file to map player roles and filenames
    config_mapping = []
    for line in config_lines:
        parts = line.strip().split(',')
        if len(parts) >= 3:
            config_mapping.append({
                'name': parts[0],
                'filename': parts[1] + '.txt',
                'role': parts[2]
            })

    # Read text files based on config mapping
    text_files = {}
    for config in config_mapping:
        filepath = os.path.join(directory, config['filename'])
        if os.path.exists(filepath):
            with open(filepath, 'r') as file:
                text_files[config['role']] = file.read()

    # Function to convert text file content to JSON object
    def text_to_json(text_files):
        json_obj = {}

        player_counter = 3
        for config in config_mapping:
            role = config['role']
            content = text_files.get(role, None)
            if content is None:
                continue

            if role == 'p1':
                player_key = 'player_1'
            elif role == 'p2':
                player_key = 'player_2'
            else:
                player_key = f'player_{player_counter}'
                player_counter += 1

            issues = content.strip().split('\n')
            json_obj[player_key] = {}
            for issue_idx, issue in enumerate(issues):
                if issue_idx == len(issues) - 1:
                    issue_key = "threshold"
                    json_obj[player_key][issue_key] = int(issue)
                else:
                    issue_key = f"issue_{issue_idx + 1}"
                    json_obj[player_key][issue_key] = list(map(int, issue.split(', ')))

        return json_obj

    # Convert text files to JSON object
    return text_to_json(text_files)

In [3]:
from itertools import product

def compute_feasibility_set(scores):
    feasibility_set = []
    players = scores.keys()

    # Generate all possible deals (Cartesian product of sub-issues)
    all_deals = list(product(
        range(len(scores['player_1']['issue_1'])),
        range(len(scores['player_1']['issue_2'])),
        range(len(scores['player_1']['issue_3'])),
        range(len(scores['player_1']['issue_4'])),
        range(len(scores['player_1']['issue_5']))
    ))

    # Check each deal
    for deal in all_deals:
        acceptable_count = 0
        player1_accepted = False
        player2_accepted = False

        for player in players:
            player_score = (
                scores[player]['issue_1'][deal[0]] +
                scores[player]['issue_2'][deal[1]] +
                scores[player]['issue_3'][deal[2]] +
                scores[player]['issue_4'][deal[3]] +
                scores[player]['issue_5'][deal[4]]
            )

            if player_score >= scores[player]['threshold']:
                acceptable_count += 1
                if player == 'player_1':
                    player1_accepted = True
                if player == 'player_2':
                    player2_accepted = True

        # Add the deal to the feasibility set if acceptable by 5 or 6 players, including player1 and player2
        if acceptable_count >= 5 and player1_accepted and player2_accepted:
            feasibility_set.append(deal)

    return feasibility_set

In [4]:
def get_pareto_efficient_deals(feasibility_set, scores):
    def dominates(deal1, deal2):
        better_or_equal = True
        strictly_better = False

        for player in scores.keys():
            score1 = (
                scores[player]['issue_1'][deal1[0]] +
                scores[player]['issue_2'][deal1[1]] +
                scores[player]['issue_3'][deal1[2]] +
                scores[player]['issue_4'][deal1[3]] +
                scores[player]['issue_5'][deal1[4]]
            )

            score2 = (
                scores[player]['issue_1'][deal2[0]] +
                scores[player]['issue_2'][deal2[1]] +
                scores[player]['issue_3'][deal2[2]] +
                scores[player]['issue_4'][deal2[3]] +
                scores[player]['issue_5'][deal2[4]]
            )

            if score1 < score2:
                better_or_equal = False
            if score1 > score2:
                strictly_better = True

        return better_or_equal and strictly_better

    pareto_efficient_deals = []

    for deal in feasibility_set:
        is_dominated = False
        for other_deal in feasibility_set:
            if dominates(other_deal, deal):
                is_dominated = True
                break

        if not is_dominated:
            pareto_efficient_deals.append(deal)

    return pareto_efficient_deals

In [5]:
def get_sparsity(scores):
    # Percentage of 0s in the scores
    total_elements = 0
    zero_count = 0
    for player in scores.keys():
        for issue in scores[player].keys():
            if issue != 'threshold':
                total_elements += len(scores[player][issue])
                zero_count += scores[player][issue].count(0)
    sparsity = (zero_count / total_elements) * 100

    return sparsity

In [6]:
import pandas as pd

def multi_set_iou(scores_dict):
    """
    Computes the multi-set IoU across all players for each issue.
    Returns a DataFrame with one row per issue and columns:
    [issue, intersection, union, iou].
    """
    players = list(scores_dict.keys())
    # Identify issues by picking keys that start with "issue_"
    issues = [k for k in scores_dict[players[0]] if k.startswith("issue_")]
    
    rows = []
    
    for issue in issues:
        # Collect the distributions for this issue across all players
        distributions = [scores_dict[player][issue] for player in players]
        
        # Intersection: sum of pointwise minima across all players
        # Union: sum of pointwise maxima across all players
        # We'll assume all players have the same length distribution for each issue.
        pointwise_minima = []
        pointwise_maxima = []
        
        # Zip across the bins for all distributions:
        # e.g. for issue_1, you might have:
        # player_1: [14, 8, 0]
        # player_2: [ 0,11, 5]
        # ...
        # We'll compute min(...) and max(...) across each bin position
        for bins_for_all_players in zip(*distributions):
            pointwise_minima.append(min(bins_for_all_players))
            pointwise_maxima.append(max(bins_for_all_players))
        
        total_min = sum(pointwise_minima)
        total_max = sum(pointwise_maxima)
        
        iou_val = total_min / total_max if total_max != 0 else 0.0
        
        rows.append({
            "issue": issue,
            "intersection": total_min,
            "union": total_max,
            "IoU": iou_val
        })
    
    df = pd.DataFrame(rows)
    # Optionally add an 'average IoU' row
    average_iou = df["IoU"].mean() if not df.empty else 0.0
    average_intersection = df["intersection"].mean() if not df.empty else 0.0
    average_union = df["union"].mean() if not df.empty else 0.0
    df.loc[len(df.index)] = {
        "issue": "Average",
        "intersection": average_intersection,
        "union": average_union,
        "IoU": average_iou
    }
    return df

# Compute multi-set IoU for each game
# multi_set_iou_dfs = {}
# for game in games:
#     print(f"Processing game: {game}")
#     data = get_data(game)
#     multi_set_iou_dfs[game] = multi_set_iou(get_data(game))
#     print(multi_set_iou_dfs[game])

In [9]:
import json
def process_game(game):
    
    # Read the game data
    json_data = get_data(game)
    #Print parsed json, indent=4
    print(json.dumps(json_data, indent=4))

    results = {}

    # Total number of deals
    for player in json_data.keys():
        total_deals = len(json_data[player]['issue_1']) * len(json_data[player]['issue_2']) * len(json_data[player]['issue_3']) * len(json_data[player]['issue_4']) * len(json_data[player]['issue_5'])
    results['total_deals'] = int(total_deals)

    # Compute feasibility set
    feasible_deals = compute_feasibility_set(json_data)
    results['feasibility_set'] = len(feasible_deals)

    # Compute R value (PI_acceptable / total_deals)
    results['R'] = results['feasibility_set'] / results['total_deals']

    # Sparsity of the game
    results['sparsity'] = get_sparsity(json_data)

    # Multi-set IoU
    multi_set_iou_df = multi_set_iou(json_data)
    results['multi_set_iou'] = multi_set_iou_df['IoU'].iloc[-1]*100

    

    # Compute Pareto efficient deals
    pareto_deals = get_pareto_efficient_deals(feasible_deals, json_data)
    results['pareto_deals'] = len(pareto_deals)

    

    # # Player 1 threshold
    # results['P1_threshold'] = json_data['player_1']['threshold']

    # # Player 2 threshold
    # results['P2_threshold'] = json_data['player_2']['threshold']

    return results

In [10]:
game = 'base'

results = process_game(game)
# print(results)
print(json.dumps(results, indent=4))

{
    "player_3": {
        "issue_1": [
            0,
            22,
            45
        ],
        "issue_2": [
            0,
            25,
            55
        ],
        "issue_3": [
            0,
            0,
            0,
            0
        ],
        "issue_4": [
            0,
            0,
            0,
            0
        ],
        "issue_5": [
            0,
            0,
            0,
            0,
            0
        ],
        "threshold": 55
    },
    "player_4": {
        "issue_1": [
            0,
            22,
            45
        ],
        "issue_2": [
            0,
            25,
            55
        ],
        "issue_3": [
            0,
            0,
            0,
            0
        ],
        "issue_4": [
            0,
            0,
            0,
            0
        ],
        "issue_5": [
            0,
            0,
            0,
            0,
            0
        ],
        "threshold": 55
    },
    "player_

In [11]:
old_data = {
    "player_3": {
        "issue_1": [
            0,
            22,
            45
        ],
        "issue_2": [
            0,
            25,
            55
        ],
        "issue_3": [
            0,
            0,
            0,
            0
        ],
        "issue_4": [
            0,
            0,
            0,
            0
        ],
        "issue_5": [
            0,
            0,
            0,
            0,
            0
        ],
        "threshold": 55
    },
    "player_4": {
        "issue_1": [
            0,
            22,
            45
        ],
        "issue_2": [
            0,
            25,
            55
        ],
        "issue_3": [
            0,
            0,
            0,
            0
        ],
        "issue_4": [
            0,
            0,
            0,
            0
        ],
        "issue_5": [
            0,
            0,
            0,
            0,
            0
        ],
        "threshold": 55
    },
    "player_5": {
        "issue_1": [
            0,
            22,
            45
        ],
        "issue_2": [
            0,
            25,
            55
        ],
        "issue_3": [
            0,
            0,
            0,
            0
        ],
        "issue_4": [
            0,
            0,
            0,
            0
        ],
        "issue_5": [
            0,
            0,
            0,
            0,
            0
        ],
        "threshold": 55
    },
    "player_1": {
        "issue_1": [
            14,
            8,
            0
        ],
        "issue_2": [
            11,
            7,
            0
        ],
        "issue_3": [
            0,
            5,
            10,
            17
        ],
        "issue_4": [
            35,
            29,
            20,
            0
        ],
        "issue_5": [
            0,
            5,
            10,
            15,
            23
        ],
        "threshold": 55
    },
    "player_2": {
        "issue_1": [
            0,
            11,
            5
        ],
        "issue_2": [
            0,
            20,
            25
        ],
        "issue_3": [
            0,
            2,
            4,
            9
        ],
        "issue_4": [
            10,
            26,
            40,
            0
        ],
        "issue_5": [
            4,
            8,
            15,
            12,
            0
        ],
        "threshold": 65
    },
    "player_6": {
        "issue_1": [
            0,
            22,
            45
        ],
        "issue_2": [
            0,
            25,
            55
        ],
        "issue_3": [
            0,
            0,
            0,
            0
        ],
        "issue_4": [
            0,
            0,
            0,
            0
        ],
        "issue_5": [
            0,
            0,
            0,
            0,
            0
        ],
        "threshold": 55
    }
}


def transform_scores(original_dict):
    """
    Transforms from the 'player_X' -> { 'issue_1': [...], ..., 'threshold': ... }
    format to the 'Name' -> { 'file_name': ..., 'role': ..., 'incentive': ..., 'scores': {...} } format.
    """

    # 1. Define how to map each "player_X" to the final names you want
    player_name_map = {
        "player_3": "Mayor",
        "player_4": "Other cities",
        "player_5": "Local Labour Union",
        "player_1": "SportCo",
        "player_2": "Department of Tourism",
        "player_6": "Environmental League",
    }

    # 2. Define each entity's "file_name", "role", and "incentive"
    #    (adjust if you have different roles or incentives)
    file_name_map = {
        "Mayor": "mayor",
        "Other cities": "other_cities",
        "Local Labour Union": "union",
        "SportCo": "SportCo",
        "Department of Tourism": "DoT",
        "Environmental League": "enviroment",
    }
    role_map = {
        "Mayor": "player",
        "Other cities": "player",
        "Local Labour Union": "player",
        "SportCo": "p1",  # per your example
        "Department of Tourism": "p2",
        "Environmental League": "player",
    }
    incentive_map = {
        "Mayor": "cooperative",
        "Other cities": "cooperative",
        "Local Labour Union": "cooperative",
        "SportCo": "greedy",
        "Department of Tourism": "greedy",
        "Environmental League": "greedy",
    }

    # 3. Map issues "issue_1"..."issue_5" to "A"..."E"
    issue_label_map = {
        "issue_1": "A",
        "issue_2": "B",
        "issue_3": "C",
        "issue_4": "D",
        "issue_5": "E",
    }

    # 4. Build the output dictionary
    transformed = {}

    for player_key, player_data in original_dict.items():
        # e.g. player_key = "player_3"
        if player_key not in player_name_map:
            # skip or handle unexpected keys
            continue

        # New label, e.g. "Mayor"
        new_label = player_name_map[player_key]

        # Prepare the sub-dict
        new_entry = {}
        new_entry["file_name"] = file_name_map[new_label]
        new_entry["role"] = role_map[new_label]
        new_entry["incentive"] = incentive_map[new_label]

        # Convert the old "threshold" -> new "min"
        min_val = player_data["threshold"]

        # Build the "scores" object: "A", "B", "C", "D", "E", plus "min"
        scores_subdict = {}
        for issue_key, issue_values in player_data.items():
            # skip the "threshold" key, we already handled it
            if issue_key.startswith("issue_"):
                # e.g. "issue_1" -> "A"
                letter = issue_label_map[issue_key]
                scores_subdict[letter] = issue_values

        # Add the "min" field
        scores_subdict["min"] = min_val

        new_entry["scores"] = scores_subdict

        transformed[new_label] = new_entry

    return transformed

transformed_data = transform_scores(old_data)

print(json.dumps(transformed_data, indent=4))

{
    "Mayor": {
        "file_name": "mayor",
        "role": "player",
        "incentive": "cooperative",
        "scores": {
            "A": [
                0,
                22,
                45
            ],
            "B": [
                0,
                25,
                55
            ],
            "C": [
                0,
                0,
                0,
                0
            ],
            "D": [
                0,
                0,
                0,
                0
            ],
            "E": [
                0,
                0,
                0,
                0,
                0
            ],
            "min": 55
        }
    },
    "Other cities": {
        "file_name": "other_cities",
        "role": "player",
        "incentive": "cooperative",
        "scores": {
            "A": [
                0,
                22,
                45
            ],
            "B": [
                0,
                25,
           

In [ ]:
games = ['base', 'base_lower_threshold_5','base_lower_threshold_10', 'base_lower_threshold_20', 'base_rewritten', 'game1', 'game2', 'game3', 'base_higher_sparsity']

results = {}
import pandas as pd
for game in games:
    results[game] = process_game(game)

results = pd.DataFrame(results).T
# Convert int columns to int and ensure float formatting
int_columns = ['total_deals', 'feasibility_set', 'pareto_deals']
results[int_columns] = results[int_columns].astype(int)
float_columns = ['sparsity', 'R', 'multi_set_iou']
results[float_columns] = results[float_columns].applymap(lambda x: round(x, 3))
print(results)

# Save results to txt file
results.to_csv('stats.txt', sep='\t')


In [ ]:
their_results = {
    "base": [81, 33, 100],
    "base_rewritten": [86, 24, 100],
    "game1": [65, 10, 85],
    "game2": [70, 40, 90],
    "game3": [86, 81, 95]
}

# Convert their_results to a DataFrame
their_results_df = pd.DataFrame(their_results, index=["5-way agreement", "6-way agreement", "Final"]).T

# Merge their_results_df with your results DataFrame on game names
merged_df = df.join(their_results_df)

# Compute correlations with R and feasibility_set
correlations = merged_df.corr()

# Extract relevant correlations
relevant_correlations = correlations.loc[["R", "feasibility_set", "sparsity"], ["5-way agreement", "6-way agreement", "Final"]]
print(relevant_correlations)

In [ ]:
import pandas as pd
import itertools

def compute_iou(distribution_a, distribution_b):
    """
    Given two lists of numbers (distribution_a, distribution_b),
    compute the Intersection over Union (IoU).
    
    IoU = (sum of pointwise minima) / (sum of pointwise maxima).
    """
    overlap = sum(min(a, b) for a, b in zip(distribution_a, distribution_b))
    union   = sum(max(a, b) for a, b in zip(distribution_a, distribution_b))
    return overlap / union if union != 0 else 0.0

def iou_dataframe(scores_dict, only_player_1_and_2=False):
    """
    Given the dictionary of player -> {issue -> [scores]},
    returns a DataFrame with one row per pair of players,
    and columns for each issue's IoU (plus optional average).
    """
    # Collect all players and issues
    players = list(scores_dict.keys())
    # Assume all players have the same issues
    issues = [k for k in scores_dict[players[0]].keys() if k.startswith("issue_")]
    #print(issues)
    # For storing results
    rows = []
    
    # Iterate over all unique pairs of players
    for player_a, player_b in itertools.combinations(players, 2):
        if only_player_1_and_2 and not ((player_a == "player_1" and player_b == "player_2") or player_a == "player_2" and player_b == "player_1"):
            continue
        row_data = {
            "player_a": player_a,
            "player_b": player_b
        }
        iou_values = []
        
        # Compute IoU for each issue
        for issue in issues:
            dist_a = scores_dict[player_a][issue]
            dist_b = scores_dict[player_b][issue]
            
            iou_val = compute_iou(dist_a, dist_b)
            row_data[f"{issue}_IoU"] = iou_val
            iou_values.append(iou_val)
        
        # Optionally compute average IoU across all issues
        row_data["mean_IoU"] = sum(iou_values) / len(iou_values) if iou_values else 0.0
        
        rows.append(row_data)
    
    # Convert the collected rows into a DataFrame
    df = pd.DataFrame(rows)
    return df


# Compute IoU DataFrame for each game
iou_dfs = {}
for game in games:
    print(f"Processing game: {game}")
    data = get_data(game)
    iou_dfs[game] = iou_dataframe(get_data(game), only_player_1_and_2=True)
    print(iou_dfs[game])

In [ ]:

def find_pareto_efficient_deals(scores):
    """
    Given a dictionary of scores in the format:
        {
          "player_name": {
            "issue_1": [...],
            "issue_2": [...],
            "issue_3": [...],
            "issue_4": [...],
            "issue_5": [...],
            "threshold": <some number>
          },
          ...
        }
    returns a list of Pareto-efficient deals.
    
    A "deal" is a tuple (i1, i2, i3, i4, i5) indicating which option index
    is chosen for each issue_1 .. issue_5. The total number of
    possible deals is the product of the number of options in each issue.
    
    Steps:
      1. Generate all possible deals (Cartesian product).
      2. For each deal, compute each player's total score.
      3. Filter deals that fail any player's threshold.
      4. Among the remaining deals, find those not dominated by any other
         (i.e., Pareto-efficient).
    """
    
    players = list(scores.keys())
    
    # --- 1) Identify how many options each issue has ---
    # We'll assume all players have the same shape for the "issue_X" lists.
    # So we just look at the first player in the dictionary.
    first_player = players[0]
    issues = [k for k in scores[first_player].keys() if k.startswith("issue_")]
    issues.sort()  # Ensure consistent ordering: issue_1, issue_2, ...
    
    # For each issue, find how many options it has (length of the list).
    num_options_per_issue = []
    for issue in issues:
        num_options_per_issue.append(len(scores[first_player][issue]))
    
    # --- 2) Generate all possible deals via Cartesian product ---
    # Each deal is a tuple like (i1, i2, ..., i5) where
    # i1 ranges 0..(len(issue_1)-1), i2 ranges 0..(len(issue_2)-1), etc.
    all_deals = itertools.product(
        *[range(n) for n in num_options_per_issue]
    )
    
    # We'll store feasibility: for each deal, what's the total score per player?
    feasible_deals = []
    
    # Precompute each player's threshold
    thresholds = {p: scores[p]["threshold"] for p in players}
    
    for deal in all_deals:
        # deal is e.g. (0, 2, 1, 3, 4)
        # We'll compute each player's total utility for this deal
        player_utilities = {}
        
        for p in players:
            total_score = 0
            for issue_idx, option_idx in enumerate(deal):
                issue_name = issues[issue_idx]  # e.g. "issue_1"
                total_score += scores[p][issue_name][option_idx]
            player_utilities[p] = total_score
        
        # --- 3) Filter out deals that fail any player's threshold ---
        # If any player's total_score < threshold, skip it
        if any(player_utilities[p] < thresholds[p] for p in players):
            continue
        
        # If we get here, the deal is feasible
        feasible_deals.append((deal, player_utilities))
    
    # --- 4) Find Pareto-efficient deals among feasible_deals ---
    # We'll define a function "dominates" that checks if dealA dominates dealB
    def dominates(dealA_utils, dealB_utils):
        """
        Return True if dealA_utils >= dealB_utils for all players
        and dealA_utils > dealB_utils for at least one player.
        """
        at_least_one_strict = False
        for p in players:
            if dealA_utils[p] < dealB_utils[p]:
                return False  # A is worse for p
            if dealA_utils[p] > dealB_utils[p]:
                at_least_one_strict = True
        return at_least_one_strict
    
    pareto_deals = []
    # We'll do a standard O(n^2) check in the feasible set. For each feasible deal,
    # we check if there's another feasible deal that dominates it.
    for i, (deal_i, utils_i) in enumerate(feasible_deals):
        dominated = False
        for j, (deal_j, utils_j) in enumerate(feasible_deals):
            if j == i:
                continue
            if dominates(utils_j, utils_i):
                dominated = True
                break
        if not dominated:
            pareto_deals.append((deal_i, utils_i))
    
    return pareto_deals

In [ ]:
# Compute Pareto-efficient deals for each game
data = get_data("base")
pareto_deals = find_pareto_efficient_deals(data)
print(f"Pareto-efficient deals for 'base' game:")
for deal, utils in pareto_deals:
    print(f"Deal: {deal} | Utilities: {utils}")
